In [ ]:
# Activate notebook environment
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()  # Install dependencies if needed

# Rigorous Ulam Method for 2D Dynamical Systems

This notebook demonstrates the rigorous Ulam method implementation for computing transfer operators of 2D dynamical systems with rigorous enclosures.

## Features

- **Rigorous interval enclosures** using IntervalArithmetic.jl
- **Adaptive bisection** for computing transition probabilities
- **Sparse matrix construction** for efficient storage
- **Additive noise** support via Minkowski sums
- **Multiple enclosure methods**: zeroth-order (pure interval) and first-order (mean value)


In [ ]:
using RigorousUlam2D
using IntervalArithmetic, IntervalBoxes
using SparseArrays
using LinearAlgebra
using Printf

## 1. Basic Interval Enclosures

First, let's demonstrate the interval enclosure methods for computing rigorous bounds on the image of a box under a map.

In [ ]:
# Define a simple 2D map: logistic map × contraction
F(x, y) = (4 * x * (1 - x), 0.5 * y)

# Create a box in phase space
box = IntervalBox(interval(0.2, 0.4), interval(0.3, 0.5))

# Compute enclosures
zeroth_enclosure = zeroth_order_enclosure(F, box)
first_enclosure = first_order_enclosure(F, box)

println("Input box: $box")
println("Zeroth-order enclosure: $zeroth_enclosure")
println("First-order enclosure:  $first_enclosure")

## 2. Partition Utilities

The package provides utilities for creating and manipulating partitions of the phase space.

In [ ]:
# Create a domain
domain = IntervalBox(interval(0.0, 1.0), interval(0.0, 1.0))

# Create a uniform partition by repeated quadrisection
boxes = [domain]
for level in 1:3  # 3 levels → 4^3 = 64 boxes
    new_boxes = IntervalBox{2, Float64, Interval{Float64}}[]
    for box in boxes
        append!(new_boxes, quadrisect(box))
    end
    boxes = new_boxes
end

println("Created partition with $(length(boxes)) boxes")
println("First few boxes:")
for (i, b) in enumerate(boxes[1:min(4, length(boxes))])
    println("  Box $i: $b, area = $(box_measure(b))")
end

## 3. Adaptive Partition Based on Dynamics

We can create an adaptive partition that refines where the dynamics requires it.

In [ ]:
# Henon-like map
function henon_map(a=1.4, b=0.3)
    return (x, y) -> (1 - a*x^2 + y, b*x)
end

H = henon_map()
domain_henon = IntervalBox(interval(-1.5, 1.5), interval(-0.5, 0.5))

# Create adaptive partition
adaptive_boxes = adaptive_partition(domain_henon, H, 
                                   initial_depth=3, 
                                   max_refinement_depth=5)

println("Adaptive partition created with $(length(adaptive_boxes)) boxes")

# Compute size statistics
sizes = [box_measure(b) for b in adaptive_boxes]
println("Box size range: [$(minimum(sizes)), $(maximum(sizes))]")
println("Mean box size: $(sum(sizes)/length(sizes))")

## 4. Computing Relative Size (Transition Probabilities)

The core algorithm computes the measure of points in one box that map into another.

In [ ]:
# Simple tent map example
tent_map(x, y) = (2 * min(x, 1-x), 2 * min(y, 1-y))

# Define source and target boxes
Q_source = IntervalBox(interval(0.0, 0.25), interval(0.0, 0.5))
Q_target = IntervalBox(interval(0.0, 0.5), interval(0.0, 1.0))

# Compute rigorous bounds on transition probability
lower, upper, uncertain = rel_size(tent_map, Q_source, Q_target, 
                                  maxdepth=10, ε=1e-4)

@printf("Transition probability bounds: [%.6f, %.6f]\n", lower/box_measure(Q_source), upper/box_measure(Q_source))
println("Uncertain boxes: $(length(uncertain))")
if !isempty(uncertain)
    uncertain_measure = sum(box_measure(b) for b in uncertain)
    @printf("Uncertain measure: %.6e (%.2f%% of source)\n", 
            uncertain_measure, 100*uncertain_measure/box_measure(Q_source))
end

## 5. Building Sparse Transition Matrices

Construct the full transfer operator as a sparse matrix.

In [ ]:
# Create a moderate-sized partition
domain_simple = IntervalBox(interval(0.0, 1.0), interval(0.0, 1.0))
partition_boxes = [domain_simple]

for _ in 1:2  # 16 boxes
    new_boxes = IntervalBox{2, Float64, Interval{Float64}}[]
    for box in partition_boxes
        append!(new_boxes, quadrisect(box))
    end
    partition_boxes = new_boxes
end

partition = UlamPartition(partition_boxes, domain_simple)

# Define a simple contracting-expanding map
baker_like(x, y) = (2*x - floor(2*x), 0.5*y + 0.25*floor(2*x))

println("Building transition matrix for $(partition.n_boxes) boxes...")
P, U = build_transition_matrix(baker_like, partition, 
                               maxdepth=8, ε=1e-4)

# Analyze the matrix
stats = estimate_sparsity(P)
println("\nMatrix statistics:")
println("  Size: $(size(P))")
println("  Non-zero entries: $(stats.nnz)")
@printf("  Sparsity: %.1f%%\n", 100*stats.sparsity)
@printf("  Avg entries per column: %.1f\n", stats.avg_per_column)

# Check column sums (should be ≈ 1 for stochastic matrix)
col_sums = [sum(P[:, j]) for j in 1:partition.n_boxes]
@printf("  Column sum range: [%.4f, %.4f]\n", minimum(col_sums), maximum(col_sums))

## 6. Noise Convolution (Minkowski Sum)

Handle systems with additive noise using Minkowski sums.

In [ ]:
# Define noise level
noise = uniform_noise_box(0.05, 2)  # ±0.05 in each direction
println("Noise box: $noise")

# Compute noisy image
test_box = IntervalBox(interval(0.4, 0.6), interval(0.4, 0.6))
deterministic_image = first_order_enclosure(baker_like, test_box)
noisy_img = noisy_image(baker_like, test_box, noise)

println("\nDeterministic image: $deterministic_image")
println("Noisy image:         $noisy_img")
println("Image expansion due to noise: $(box_measure(noisy_img) / box_measure(deterministic_image))×")

## 7. Transition Matrix with Noise

Build a transition matrix for a noisy dynamical system.

In [ ]:
# Use a smaller partition for demonstration
small_boxes = quadrisect(domain_simple)
for _ in 1:1
    new_boxes = IntervalBox{2, Float64, Interval{Float64}}[]
    for box in small_boxes
        append!(new_boxes, quadrisect(box))
    end
    small_boxes = new_boxes
end

small_partition = UlamPartition(small_boxes, domain_simple)
noise_small = uniform_noise_box(0.02, 2)

println("Building noisy transition matrix...")
P_noisy, U_noisy = build_noisy_transition_matrix(baker_like, small_partition, 
                                                 noise_small, maxdepth=6, ε=1e-3)

println("\nNoisy matrix statistics:")
stats_noisy = estimate_sparsity(P_noisy)
println("  Non-zero entries: $(stats_noisy.nnz)")
@printf("  Sparsity: %.1f%%\n", 100*stats_noisy.sparsity)

# Compare with deterministic
P_det, _ = build_transition_matrix(baker_like, small_partition, maxdepth=6, ε=1e-3)
stats_det = estimate_sparsity(P_det)

println("\nNoise effect on sparsity:")
println("  Deterministic nnz: $(stats_det.nnz)")
println("  Noisy nnz:         $(stats_noisy.nnz)")
println("  Increase:          $(stats_noisy.nnz - stats_det.nnz) entries")

## 8. Computing Invariant Measures

Find the invariant density by computing the leading eigenvector.

In [ ]:
# Normalize the transition matrix
P_normalized = copy(P_noisy)
col_sums_norm = normalize_columns!(P_normalized)

println("Column normalization:")
@printf("  Sum range after normalization: [%.6f, %.6f]\n", 
        minimum([sum(P_normalized[:, j]) for j in 1:size(P_normalized, 2)]),
        maximum([sum(P_normalized[:, j]) for j in 1:size(P_normalized, 2)]))

# Compute eigenvalues (using dense for small matrices)
P_dense = Matrix(P_normalized)
λ = eigvals(P_dense)
λ_sorted = sort(abs.(λ), rev=true)

println("\nSpectrum:")
println("  Leading eigenvalues: $(λ_sorted[1:min(5, length(λ_sorted))])")
println("  Spectral gap: $(1 - λ_sorted[2])")

## 9. Summary

This notebook demonstrated:

1. **Rigorous interval enclosures** for 2D maps using zeroth and first-order methods
2. **Partition utilities** including quadrisection and adaptive refinement  
3. **Relative size computation** using adaptive bisection for rigorous transition probabilities
4. **Sparse matrix construction** for efficient transfer operator representation
5. **Noise handling** via Minkowski sums for stochastic systems
6. **Spectral analysis** of the transfer operator

### Key Features

- All computations are **mathematically rigorous** using interval arithmetic
- Matrices are **sparse** for computational efficiency
- **Adaptive methods** reduce computational cost while maintaining accuracy
- **Noise support** enables analysis of stochastic systems

### Next Steps

- Implement higher-order enclosures (centered forms, Taylor models)
- Add visualization tools for partitions and invariant measures
- Support for non-uniform partitions
- Monte Carlo comparison for validation
- Extensions to higher-dimensional systems